# Hand-made GoCCvA alignment examples

This notebook demonstrates goal-oriented alignment visualization for a case study.

- Models are loaded in the notebook.
- Handmade test cases are defined inline.
- Target rows are computed from the loaded goal model using `goal_model.compute_target_sets(target)`.
- Rendering is delegated to reusable helper functions in `Ui.goccva_ui`.

## Setup & Imports

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Adjust path if needed
if Path.cwd().name == "work-GoCCvA-2026":
    project_dir = Path.cwd()
else:
    project_dir = Path("work-GoCCvA-2026").resolve()

sys.path.insert(0, str(project_dir))

from Ui.goccva_ui import (
    render_all_goal_oriented_alignments,
    build_target_computation_func,
)
# Uncomment or adapt these imports according to your repository structure
from Semantics.istar_processor import read_istar_model
from Semantics.petri_net_processor import read_petri_net
from Semantics.event_mapping_from_csv import read_event_mapping_csv

In [2]:
import importlib
import Ui.goccva_ui as ghui

importlib.reload(ghui)

<module 'Ui.goccva_ui' from '/Users/huba/git/dtu_projects/kogi/work-GoCCvA-2026/Ui/goccva_ui.py'>

## Load Models

Load the goal model, process model, and mapping used by the case study.

In [3]:
# Paths used in the GoCCvA repository
goal_model_path = project_dir / "content/gm.txt"
process_model_path = project_dir / "content/pm.pnml"
mapping_path = project_dir / "content/mapping_qualified.csv"

# Load your models here. Keep the exact reader names aligned with your repository.
# Example:
goal_model = read_istar_model(str(goal_model_path))
petri_net = read_petri_net(str(process_model_path))
activity_mapping = read_event_mapping_csv(str(mapping_path))


## Target Configuration

Define the target requirements to be evaluated. The make, break, and non-related sets are computed from the loaded goal model.

In [4]:
targets = [
    "Data access provided",
    "(Hospital Officer) data easily accesible",
]

target_func = build_target_computation_func(
    goal_model=goal_model,
    targets=targets,
)

## Target Row Computation from the Goal Model

This helper converts the make, break, and non-related sets into the rows used by the alignment visualization.

For each target and each activity in a trace:

- `M` means the activity belongs to the target make set.
- `B` means the activity belongs to the target break set.
- `NR` means the activity belongs to the target non-related set.
- `ND` is used as a fallback when the activity is not classified for the selected target.

The marking is updated as follows:

- `M` changes the target state to `S`.
- `B` changes the target state to `D`.
- `NR` and `ND` preserve the previous state.

## Handmade Test Cases

Define the traces used to illustrate the six combinations of alignment quality and target fulfillment.

In [5]:
handmade_cases = [
    {
        "requested_case": "optimal alignment + strong fulfilled",
        "trace": [
            "Verify identity",
            "Identity verified",
            "Format Data Special Needs",
            "Provide Records",
        ],
        "why": "The trace can be aligned with minimum cost and ends with the selected targets satisfied.",
    },
    {
        "requested_case": "optimal alignment + weak fulfilled",
        "trace": [
            "Verify identity",
            "Identity verified",
            "Format Data Special Needs",
            "Format Data Regular Needs",
            "Format Data Special Needs",
            "Provide Records",
        ],
        "why": (
            "The trace reaches target satisfaction after previously breaking one selected target, "
            "so the final fulfillment is weak rather than strong."
        ),
    },
    {
        "requested_case": "optimal alignment + non-fulfilled",
        "trace": [
            "Verify identity",
            "Identity verified",
            "Format Data Regular Needs",
            "Provide Records",
        ],
        "why": (
            "The trace follows the regular-needs branch and leaves the accessibility target denied."
        ),
    },
    {
        "requested_case": "non-optimal alignment + strong fulfilled",
        "trace": [
            "Verify identity",
            "Identity verified",
            "Format Data Special Needs",
            "Provide Records",
            "Manual accessibility check",
        ],
        "why": (
            "The trace includes additional behavior after a path that satisfies the selected targets."
        ),
    },
    {
        "requested_case": "non-optimal alignment + weak fulfilled",
        "trace": [
            "Verify identity",
            "Identity verified",
            "Format Data Special Needs",
            "Format Data Regular Needs",
            "Format Data Special Needs",
            "Provide Records",
        ],
        "why": (
            "The trace satisfies accessibility, breaks it with regular formatting, "
            "and then satisfies it again before records are provided."
        ),
    },
    {
        "requested_case": "non-optimal alignment + non-fulfilled",
        "trace": [
            "Verify identity",
            "Identity verified",
            "Format Data Special Needs",
            "Format Data Regular Needs",
            "Provide Records",
        ],
        "why": "The trace contains behavior that leaves one selected target denied at the end.",
    },
]

print(f"Defined {len(handmade_cases)} test cases")

Defined 6 test cases


## Activity Abbreviations

In [6]:
activity_abbreviations = {
    "Verify identity": "vi",
    "Identity verified": "iv",
    "Format Data Special Needs": "fs",
    "Format Data Regular Needs": "fr",
    "Provide Records": "pr",
    "Manual accessibility check": "ma",
}

print("Activity abbreviations configured")

Activity abbreviations configured


## Render Visualization

In [7]:
render_all_goal_oriented_alignments(
    cases=handmade_cases,
    activity_abbreviations=activity_abbreviations,
    target_computation_func=target_func,
    title="Goal-oriented Process Alignment Examples (GoCCvA Case Study)",
)

# Goal-oriented Process Alignment Examples (GoCCvA Case Study)

The following cases illustrate combinations of alignment class and target-fulfilment class. Each table separates the process alignment row from the target-specific interpretation rows.